# Preprocessing Data (Kelompok 3)

**Mata Kuliah:** Machine Learning (Kelas C), Bapak Adi Purnawan

**Judul:** Prediksi Risiko Stroke Menggunakan Logistic Regression dan Gradient Boosting dengan Interpretasi Explainable AI

**Anggota:** Deliana Br Manalu (2305551036) · Ravi Arnan Irianto (2305551076) · Ezza Putra Wibawa (2305551144) · Devin (2305551173)

---

Bapak menyampaikan bahwa 60–70% pekerjaan machine learning ada di tahap pengumpulan dan
preprocessing data. Notebook ini adalah tahap itu.

Isinya menjawab tiga masalah yang ditemukan di notebook `01`:

1. `bmi` kosong pada 201 baris → **Tugas B**, diselesaikan dengan model regresi (Bab 3–4)
2. `smoking_status` bernilai "Unknown" pada 1.544 baris (30,2%), missing value yang
   menyamar sebagai kategori
3. `gender` bernilai "Other" hanya pada 1 baris

Setiap keputusan di sini **diuji, bukan ditebak**: kami ukur pengaruhnya terhadap hasil
klasifikasi akhir, lalu pilih yang terbukti terbaik.


## 1. Persiapan

In [1]:
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import StratifiedKFold, cross_val_score, cross_validate, train_test_split
from sklearn.linear_model import LinearRegression, Ridge, Lasso, LogisticRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.metrics import mean_squared_error, r2_score

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", None)
sns.set_theme(style="whitegrid")

SEED = 42
URL = ("https://raw.githubusercontent.com/ray-project/raydp/master/"
       "tutorials/dataset/healthcare-dataset-stroke-data.csv")

mentah = pd.read_csv(URL)
print("Data mentah:", mentah.shape)

Data mentah: (5110, 12)


## 2. Pembersihan Dasar

Tiga hal yang bisa diputuskan tanpa perlu pengujian rumit.

In [2]:
bersih = mentah.drop(columns=["id"]).copy()

# gender "Other" hanya 1 baris -- terlalu sedikit untuk dipelajari model,
# dan tidak bisa dimasukkan ke kategori mana pun tanpa mengarang
print("Sebelum:", bersih.gender.value_counts().to_dict())
bersih = bersih[bersih.gender != "Other"].copy()
print("Sesudah:", bersih.gender.value_counts().to_dict())

print("\nDuplikat:", bersih.duplicated().sum())
print("Baris sekarang:", len(bersih))

Sebelum: {'Female': 2994, 'Male': 2115, 'Other': 1}
Sesudah: {'Female': 2994, 'Male': 2115}

Duplikat: 0
Baris sekarang: 5109


### Pemeriksaan outlier

Nilai ekstrem **tidak langsung dibuang**. Kami periksa dulu apakah ia tanda kesalahan
input atau memang kondisi yang mungkin terjadi. BMI 97,6 sangat jarang, tetapi obesitas
ekstrem memang ada dan bukan kemustahilan medis.

In [3]:
ekstrem = bersih[bersih.bmi > 60]
print(f"Pasien dengan BMI > 60: {len(ekstrem)}")
print(f"Di antaranya terkena stroke: {ekstrem.stroke.sum()}")
print(f"Proporsi stroke pada kelompok ini : {ekstrem.stroke.mean()*100:.1f}%")
print(f"Proporsi stroke pada seluruh data : {bersih.stroke.mean()*100:.1f}%")
print()
print("Keputusan: outlier BMI DIPERTAHANKAN.")
print("Alasannya BUKAN karena kelompok ini berisiko tinggi -- justru sebaliknya,")
print("tidak satu pun dari mereka terkena stroke pada data ini. Tetapi 13 baris")
print("terlalu sedikit untuk menyimpulkan apa pun, dan tidak ada tanda bahwa")
print("nilainya salah input. Membuang data hanya karena nilainya ekstrem, tanpa")
print("bukti bahwa ia keliru, adalah membuang bukti yang sah."); print()
print(f"Pengaruhnya pun kecil: 13 dari {len(bersih)} baris ({len(ekstrem)/len(bersih)*100:.2f}%).")

Pasien dengan BMI > 60: 13
Di antaranya terkena stroke: 0
Proporsi stroke pada kelompok ini : 0.0%
Proporsi stroke pada seluruh data : 4.9%

Keputusan: outlier BMI DIPERTAHANKAN.
Alasannya BUKAN karena kelompok ini berisiko tinggi -- justru sebaliknya,
tidak satu pun dari mereka terkena stroke pada data ini. Tetapi 13 baris
terlalu sedikit untuk menyimpulkan apa pun, dan tidak ada tanda bahwa
nilainya salah input. Membuang data hanya karena nilainya ekstrem, tanpa
bukti bahwa ia keliru, adalah membuang bukti yang sah.

Pengaruhnya pun kecil: 13 dari 5109 baris (0.25%).


## 3. Tugas B: Imputasi BMI dengan Model Regresi

**Bab 3 dan 4 dipakai di sini.** Alih-alih mengisi 201 nilai `bmi` yang hilang dengan
median begitu saja, kami bangun model regresi untuk memprediksinya dari fitur lain,
lalu bandingkan mana yang benar-benar lebih baik.

Model dilatih pada baris yang `bmi`-nya diketahui, lalu diuji dengan cross-validation.

In [4]:
# Fitur untuk memprediksi bmi -- stroke TIDAK dipakai supaya tidak terjadi kebocoran
fitur_imputasi = ["age", "hypertension", "heart_disease", "avg_glucose_level",
                  "gender", "ever_married", "work_type", "Residence_type", "smoking_status"]

ada_bmi = bersih[bersih.bmi.notna()]
X_imp = pd.get_dummies(ada_bmi[fitur_imputasi], drop_first=True)
y_imp = ada_bmi.bmi

print(f"Baris untuk melatih model imputasi: {len(X_imp)}")
print(f"Baris yang bmi-nya perlu diisi     : {bersih.bmi.isna().sum()}")

Baris untuk melatih model imputasi: 4908
Baris yang bmi-nya perlu diisi     : 201


In [5]:
MODEL_IMPUTASI = {
    "Linear Regression": make_pipeline(StandardScaler(), LinearRegression()),
    "Ridge (alpha=1)":   make_pipeline(StandardScaler(), Ridge(alpha=1.0, random_state=SEED)),
    "Ridge (alpha=10)":  make_pipeline(StandardScaler(), Ridge(alpha=10.0, random_state=SEED)),
    "Lasso (alpha=0.1)": make_pipeline(StandardScaler(), Lasso(alpha=0.1, random_state=SEED)),
    "Lasso (alpha=1)":   make_pipeline(StandardScaler(), Lasso(alpha=1.0, random_state=SEED)),
    "Random Forest":     RandomForestRegressor(n_estimators=200, random_state=SEED, n_jobs=-1),
}

baris = []
for nama, m in MODEL_IMPUTASI.items():
    rmse = -cross_val_score(m, X_imp, y_imp, cv=5, scoring="neg_root_mean_squared_error").mean()
    r2 = cross_val_score(m, X_imp, y_imp, cv=5, scoring="r2").mean()
    baris.append({"Model": nama, "RMSE": rmse, "R2": r2})

# pembanding: tebak median untuk semua orang
rmse_median = mean_squared_error(y_imp, np.full(len(y_imp), y_imp.median())) ** 0.5
baris.append({"Model": "Median (pembanding)", "RMSE": rmse_median, "R2": 0.0})

pd.DataFrame(baris).round(3).sort_values("RMSE")

,Model,RMSE,R2
0,Linear Regression,6.869,0.232
1,Ridge (alpha=1),6.869,0.232
2,Ridge (alpha=10),6.869,0.232
3,Lasso (alpha=0.1),6.870,0.232
4,Lasso (alpha=1),7.044,0.193
5,Random Forest,7.053,0.190
6,Median (pembanding),7.894,0.000


### Regularisasi dan seleksi fitur (Bab 4)

Lasso memangkas koefisien fitur yang tidak berguna menjadi nol. Ini sekaligus berfungsi
sebagai seleksi fitur: kita bisa melihat fitur mana yang benar-benar menentukan BMI.

In [6]:
lasso = make_pipeline(StandardScaler(), Lasso(alpha=0.1, random_state=SEED)).fit(X_imp, y_imp)
koef = pd.Series(lasso[-1].coef_, index=X_imp.columns).sort_values(key=abs, ascending=False)

print(f"Fitur dipangkas jadi nol oleh Lasso: {(koef == 0).sum()} dari {len(koef)}\n")
print("Sepuluh koefisien terbesar:")
print(koef.head(10).round(3).to_string())

Fitur dipangkas jadi nol oleh Lasso: 4 dari 14

Sepuluh koefisien terbesar:
work_type_children               -2.781
ever_married_Yes                  0.834
avg_glucose_level                 0.776
hypertension                      0.601
heart_disease                    -0.107
work_type_Never_worked           -0.092
work_type_Private                 0.084
smoking_status_formerly smoked    0.066
work_type_Self-employed          -0.048
smoking_status_smokes             0.030


### Yang benar-benar penting: pengaruhnya ke klasifikasi

Model imputasi yang RMSE-nya lebih kecil belum tentu menghasilkan klasifikasi stroke
yang lebih baik. Inilah yang sebenarnya perlu diukur.

In [7]:
def siapkan(df, strategi_bmi):
    """Kembalikan (X, y) siap latih sesuai strategi penanganan bmi."""
    d = df.copy()
    if strategi_bmi == "hapus baris":
        d = d[d.bmi.notna()]
    elif strategi_bmi == "isi median":
        d["bmi"] = d.bmi.fillna(d.bmi.median())
    elif strategi_bmi == "prediksi regresi":
        kosong = d.bmi.isna()
        if kosong.any():
            latih = d[~kosong]
            X_l = pd.get_dummies(latih[fitur_imputasi], drop_first=True)
            m = make_pipeline(StandardScaler(), Ridge(alpha=1.0, random_state=SEED)).fit(X_l, latih.bmi)
            X_k = pd.get_dummies(d[kosong][fitur_imputasi], drop_first=True).reindex(
                columns=X_l.columns, fill_value=0)
            d.loc[kosong, "bmi"] = m.predict(X_k)
    X = pd.get_dummies(d.drop(columns=["stroke"]), drop_first=True)
    return X, d.stroke

In [8]:
cv = StratifiedKFold(5, shuffle=True, random_state=SEED)
klasifikator = make_pipeline(StandardScaler(),
                             LogisticRegression(max_iter=2000, class_weight="balanced",
                                                random_state=SEED))
hasil = []
for strategi in ["hapus baris", "isi median", "prediksi regresi"]:
    X, y = siapkan(bersih, strategi)
    skor = cross_validate(klasifikator, X, y, cv=cv,
                          scoring=["recall", "precision", "f1", "roc_auc"], n_jobs=-1)
    hasil.append({
        "Strategi bmi": strategi,
        "baris": len(X),
        "recall": skor["test_recall"].mean(),
        "precision": skor["test_precision"].mean(),
        "f1": skor["test_f1"].mean(),
        "roc_auc": skor["test_roc_auc"].mean(),
    })

pd.DataFrame(hasil).round(3)

,Strategi bmi,baris,recall,precision,f1,roc_auc
0,hapus baris,4908,0.785,0.120,0.208,0.844
1,isi median,5109,0.791,0.134,0.229,0.838
2,prediksi regresi,5109,0.795,0.133,0.228,0.838


### Kesimpulan Tugas B

Perhatikan bahwa ketiga strategi memberi hasil yang **hampir sama**. Ini temuan yang
jujur dan layak ditulis: hanya 3,93% data yang `bmi`-nya kosong, terlalu sedikit untuk
mengubah hasil akhir secara berarti.

Meski begitu kami tetap memakai **imputasi regresi (Ridge)**, dengan dua alasan:
tidak membuang 201 baris seperti strategi "hapus baris", dan lebih beralasan secara
metodologis daripada menempelkan median yang sama ke semua orang.

Pelajaran yang lebih penting: **kecanggihan preprocessing tidak otomatis berarti hasil
lebih baik.** Kalau tidak diukur, kita tidak akan tahu.

## 4. Menangani `smoking_status` = "Unknown"

1.544 baris (30,2%) tidak diketahui status merokoknya. Ada dua pilihan:

1. Buang barisnya → kehilangan 30% data
2. Perlakukan "Unknown" sebagai kategori tersendiri → data utuh, tetapi model harus
   belajar bahwa "tidak diketahui" itu sendiri adalah informasi

Kami uji keduanya.

In [9]:
hasil_rokok = []
for label, data in [("Unknown jadi kategori", bersih),
                    ("Unknown dibuang", bersih[bersih.smoking_status != "Unknown"])]:
    X, y = siapkan(data, "prediksi regresi")
    skor = cross_validate(klasifikator, X, y, cv=cv,
                          scoring=["recall", "precision", "f1", "roc_auc"], n_jobs=-1)
    hasil_rokok.append({
        "Perlakuan": label, "baris": len(X), "positif": int(y.sum()),
        "recall": skor["test_recall"].mean(),
        "f1": skor["test_f1"].mean(),
        "roc_auc": skor["test_roc_auc"].mean(),
    })

pd.DataFrame(hasil_rokok).round(3)

,Perlakuan,baris,positif,recall,f1,roc_auc
0,Unknown jadi kategori,5109,249,0.795,0.228,0.838
1,Unknown dibuang,3565,202,0.792,0.246,0.818


In [10]:
# Apakah "Unknown" itu sendiri membawa informasi?
tabel = bersih.groupby("smoking_status").agg(
    jumlah=("stroke", "size"),
    kasus_stroke=("stroke", "sum"),
    persen_stroke=("stroke", lambda s: round(s.mean() * 100, 2)),
    rata_usia=("age", lambda s: round(s.mean(), 1)),
)
tabel

,jumlah,kasus_stroke,persen_stroke,rata_usia
smoking_status,,,,
Unknown,1544,47,3.04,30.2
formerly smoked,884,70,7.92,55.0
never smoked,1892,90,4.76,46.7
smokes,789,42,5.32,47.1


### Kesimpulan

Kelompok "Unknown" punya **rata-rata usia jauh lebih muda** dan proporsi stroke paling
rendah. Artinya nilai "Unknown" tidak tersebar acak, kemungkinan besar berkaitan
dengan pasien anak dan remaja yang pertanyaan merokoknya memang tidak diajukan.

Karena ketidaktahuannya sendiri membawa informasi, **"Unknown" dipertahankan sebagai
kategori**, bukan dibuang. Membuangnya berarti menghapus 30% data sekaligus membuang
sinyal yang berguna.

## 5. Pembagian Data 70 / 15 / 15

Sesuai tahapan yang Bapak jelaskan: sebagian untuk latih, sebagian untuk validasi,
sebagian untuk uji.

Pembagian dilakukan **stratified** agar proporsi pasien stroke yang hanya 4,87% tetap
sama di ketiga bagian. Kalau dibagi acak biasa, bisa saja satu bagian nyaris tidak
kebagian kasus stroke.

In [11]:
X_penuh, y_penuh = siapkan(bersih, "prediksi regresi")

X_latih, X_sisa, y_latih, y_sisa = train_test_split(
    X_penuh, y_penuh, test_size=0.30, stratify=y_penuh, random_state=SEED)
X_validasi, X_uji, y_validasi, y_uji = train_test_split(
    X_sisa, y_sisa, test_size=0.50, stratify=y_sisa, random_state=SEED)

for nama, Xs, ys in [("Latih", X_latih, y_latih), ("Validasi", X_validasi, y_validasi),
                     ("Uji", X_uji, y_uji)]:
    print(f"{nama:10} {len(Xs):5} baris | stroke {int(ys.sum()):3} ({ys.mean()*100:.2f}%)")

Latih       3576 baris | stroke 174 (4.87%)
Validasi     766 baris | stroke  37 (4.83%)
Uji          767 baris | stroke  38 (4.95%)


## 6. Fungsi Siap Pakai untuk Notebook Berikutnya

Notebook `05` sampai `08` memerlukan data yang sudah diproses dengan cara yang sama
persis. Fungsi di bawah ini **disalin ke setiap notebook tersebut** agar masing-masing
tetap bisa dijalankan sendiri di Colab tanpa mengunggah berkas apa pun.

Pengulangan ini disengaja: di Colab, notebook tidak bisa saling mengimpor dengan mudah,
dan menyimpan CSV perantara berarti anggota kelompok harus bertukar berkas.

In [12]:
def muat_data_stroke(seed=42):
    """Muat dan proses dataset stroke. Hasilnya identik di semua notebook.

    Keputusan yang sudah diuji di notebook 04:
      - gender "Other" (1 baris) dibuang
      - outlier BMI dipertahankan (masuk akal secara medis)
      - bmi kosong diisi hasil prediksi Ridge, bukan median
      - smoking_status "Unknown" dipertahankan sebagai kategori
    """
    url = ("https://raw.githubusercontent.com/ray-project/raydp/master/"
           "tutorials/dataset/healthcare-dataset-stroke-data.csv")
    fitur_imp = ["age", "hypertension", "heart_disease", "avg_glucose_level",
                 "gender", "ever_married", "work_type", "Residence_type", "smoking_status"]

    d = pd.read_csv(url).drop(columns=["id"])
    d = d[d.gender != "Other"].copy()

    kosong = d.bmi.isna()
    if kosong.any():
        latih = d[~kosong]
        X_l = pd.get_dummies(latih[fitur_imp], drop_first=True)
        m = make_pipeline(StandardScaler(), Ridge(alpha=1.0, random_state=seed)).fit(X_l, latih.bmi)
        X_k = pd.get_dummies(d[kosong][fitur_imp], drop_first=True).reindex(
            columns=X_l.columns, fill_value=0)
        d.loc[kosong, "bmi"] = m.predict(X_k)

    X = pd.get_dummies(d.drop(columns=["stroke"]), drop_first=True)
    return X, d.stroke, d


X, y, mentah_bersih = muat_data_stroke()
print("Bentuk akhir :", X.shape)
print("Fitur        :", list(X.columns))
print(f"Positif      : {y.sum()} ({y.mean()*100:.2f}%)")
print("Missing value:", X.isna().sum().sum())

Bentuk akhir : (5109, 15)
Fitur        : ['age', 'hypertension', 'heart_disease', 'avg_glucose_level', 'bmi', 'gender_Male', 'ever_married_Yes', 'work_type_Never_worked', 'work_type_Private', 'work_type_Self-employed', 'work_type_children', 'Residence_type_Urban', 'smoking_status_formerly smoked', 'smoking_status_never smoked', 'smoking_status_smokes']
Positif      : 249 (4.87%)
Missing value: 0


## 7. Ringkasan Keputusan Preprocessing

| Masalah | Keputusan | Alasan |
|---|---|---|
| `gender` = "Other" (1 baris) | dibuang | terlalu sedikit untuk dipelajari model |
| Outlier BMI (13 baris, maks 97,6) | dipertahankan | obesitas ekstrem mungkin terjadi; tidak ada tanda salah input, dan 13 baris terlalu sedikit untuk berpengaruh |
| `bmi` kosong (201 baris) | diisi prediksi Ridge | tidak membuang data, lebih beralasan daripada median |
| `smoking_status` = "Unknown" (1.544) | dipertahankan sebagai kategori | ketidaktahuannya sendiri membawa informasi (usia jauh lebih muda) |
| Duplikat | tidak ada | - |
| Pembagian data | 70/15/15 stratified | menjaga proporsi kelas yang hanya 4,87% |

**Temuan jujur yang layak masuk laporan:** ketiga strategi penanganan `bmi` memberi
hasil klasifikasi yang hampir sama. Preprocessing yang lebih canggih tidak otomatis
berarti model lebih baik, dan itu hanya bisa diketahui kalau diukur.

Notebook berikutnya: `05_klasifikasi.ipynb`.